# Fine-grained tool calling (streaming)

Stream Claude's response while using tools, and optionally disable server-side JSON validation of tool arguments via the fine-grained streaming beta.
The `save_article` tool here is **user-implemented** (you own both the schema and the handler); the focus of this notebook is the streaming layer around tool use.

## What's different vs `1_tool_use`

- Uses `chat_stream` instead of `chat`: returns the `client.beta.messages.stream(...)` context manager.
  Inside the `with` block the loop yields events (`text`, `content_block_start`, `input_json`, `content_block_stop`); call `stream.get_final_message()` to assemble the full `Message`.
- Supports `tool_choice` to force Claude to call a specific tool.
  `tool_choice` is a per-request constraint sent on **every** API call, not just the first.
  So if we loop with it still set, after the `tool_result` Claude is forced to call that same tool again — and again, forever — because the request says "you must call this tool now".
  To avoid that we `break` after one iteration.
  Side effect: with `tool_choice` set you only see one assistant turn (the tool call). There is no follow-up text turn after the `tool_result` because the loop exits. Drop `tool_choice` (or set it only on the first turn) if you want Claude to reply to the tool result in free text.
- `betas=["fine-grained-tool-streaming-2025-05-14"]` disables JSON validation and sends tool-input chunks the moment they are generated.
  Faster UX but the client has to tolerate invalid JSON.

Model: `claude-sonnet-4-6`.


In [21]:
import json

from chat_tools import add_assistant_message, add_user_message, chat_stream
from models import MODEL_SONNET

In [22]:
def save_article(**kwargs):
    return "Article saved!"


save_article_schema = {
    "name": "save_article",
    "description": "Saves a scholarly journal article",
    "input_schema": {
        "type": "object",
        "properties": {
            "abstract": {"type": "string", "description": "Abstract of the article. One short sentence max"},
            "meta": {
                "type": "object",
                "properties": {
                    "word_count": {"type": "integer", "description": "Word count"},
                    "review": {"type": "string", "description": "Eight sentence review of the paper"},
                },
                "required": ["word_count", "review"],
            },
        },
        "required": ["abstract", "meta"],
    },
}

In [23]:
def run_tool(tool_name, tool_input):
    if tool_name == "save_article":
        return save_article(**tool_input)
    raise ValueError(f"Unknown tool: {tool_name}")


def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }
        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

In [24]:
def run_conversation(messages, tools=None, tool_choice=None, fine_grained=False):
    while True:
        with chat_stream(
            messages,
            model=MODEL_SONNET,
            tools=tools,
            tool_choice=tool_choice,
            betas=["fine-grained-tool-streaming-2025-05-14"] if fine_grained else None,
        ) as stream:
            for chunk in stream:
                if chunk.type == "text":
                    print(chunk.text, end="")
                if chunk.type == "content_block_start" and chunk.content_block.type == "tool_use":
                    print(f'\n>>> Tool Call: "{chunk.content_block.name}"')
                if chunk.type == "input_json" and chunk.partial_json:
                    print(chunk.partial_json, end="")
                if chunk.type == "content_block_stop":
                    print("\n")

            response = stream.get_final_message()

        add_assistant_message(messages, response)

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

        if tool_choice:
            break

    print("---\n")
    return messages

In [25]:
messages = []
add_user_message(messages, "Create and save a fake computer science article.")

run_conversation(
    messages,
    tools=[save_article_schema],
    tool_choice={"type": "tool", "name": "save_article"},
    fine_grained=True,
)


>>> Tool Call: "save_article"
{"abstract": "This paper introduces QuantumNet, a novel hybrid quantum-classical neural network architecture that achieves state-of-the-art performance on large-scale image classification benchmarks while reducing computational complexity by an order of magnitude.", "meta": {"word_count": 8432, "review": "This paper presents QuantumNet, an innovative architecture that merges quantum computing principles with classical deep learning frameworks, a timely contribution given the rapid advances in both fields. The authors provide a thorough theoretical foundation, grounding their approach in well-established quantum gate operations and variational circuit design. The experimental results are compelling, demonstrating a 10x reduction in floating-point operations compared to ResNet-152 while achieving a 97.3% top-1 accuracy on ImageNet. One of the paper's greatest strengths is its reproducibility section, which includes detailed hyperparameter settings, circuit 

[{'role': 'user',
  'content': 'Create and save a fake computer science article.'},
 {'role': 'assistant',
  'content': ParsedBetaMessage(id='msg_011CgBrTgCzaWKJYRLktr56T', container=None, content=[BetaToolUseBlock(id='toolu_018FdjCM2uriHGBBRoyNLbeD', input={'abstract': 'This paper introduces QuantumNet, a novel hybrid quantum-classical neural network architecture that achieves state-of-the-art performance on large-scale image classification benchmarks while reducing computational complexity by an order of magnitude.', 'meta': {'word_count': 8432, 'review': "This paper presents QuantumNet, an innovative architecture that merges quantum computing principles with classical deep learning frameworks, a timely contribution given the rapid advances in both fields. The authors provide a thorough theoretical foundation, grounding their approach in well-established quantum gate operations and variational circuit design. The experimental results are compelling, demonstrating a 10x reduction in f